# Prefix bridge: the encoder's output fed once, as input, to Gemma + LoRA

```
"<voice>: <English sentence>"  →  ModernBERT-base + LoRA  →  bridge (MLP 768 → 2560)  →  S prefix vectors   [GPU 0]
                                                                                               │
[<bos>] + [prefix vectors] + target  →  Gemma 3 4B IT (4-bit) + LoRA  →  Hinglish  <end_of_turn>            [GPU 1]
```

**What changes from `8-bridge-pretraining/`:** no cross-attention blocks and no gates. The bridge turns each encoder token into a vector in Gemma's own input-embedding space, and these vectors are placed after `<bos>` as a **prefix**: Gemma reads them with its **own pretrained attention** at every layer, as it reads any input. The bridge is a 2-layer MLP (as in LLaVA) with its output scaled to match Gemma's token embeddings. Trained: encoder LoRA, bridge (≈ 8.5 M parameters instead of ≈ 210 M of cross-attention), decoder LoRA. As before, persona and sentence reach Gemma only through the encoder; there is no text prompt.

| Stage | Data | Encoder input | Pairs |
|---|---|---|---|
| **1. Translation pre-training** | Hinglish-TOP (Apache-2.0) + PHINC (CC BY 4.0) | `Plain: <English>` | `CORPUS_TOTAL` in Cell 3 (default 12,000; **0 skips stage 1**) |
| **2. Persona fine-tuning** | v4 merged | `Don: <English>` (etc.) | 768, up to 6 epochs |

Corpora, cleaning and the leakage guard are as in `8-bridge-pretraining/` (PHINC: Srivastava & Singh, WNUT 2020, CC BY 4.0). Reported: held-out corpus loss and translations after stage 1; v4 validation, test and out-of-domain translations after stage 2 (chrF, stopping, distinct outputs, examples).

In [ ]:
# Cell 0 — Hugging Face token (Gemma 3 is gated). Clear it before saving or sharing the notebook.

HF_TOKEN = ""

## Cell 1 — Setup, pinned revisions

Two GPUs (Kaggle: *GPU T4 × 2*). Models and both corpora are pinned to the revision found now.

In [ ]:
# Cell 1 — Packages, settings, pinned revisions

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q "transformers==5.17.0" "peft==0.21.2" -U bitsandbytes

import json, os, random, re, statistics, time, urllib.request
from collections import Counter

import pandas as pd
import peft
import torch
import torch.nn as nn
import transformers
from huggingface_hub import HfApi, login

ENCODER, DECODER = "answerdotai/ModernBERT-base", "google/gemma-3-4b-it"
TOP_REPO, PHINC_REPO = "rvv-karma/English-Hinglish-TOP", "veezbo/phinc"
DATA = ("https://raw.githubusercontent.com/skamalj/llms-the-hard-way/hindi-decoder-compare/"
        "project4-hindi-decoder/data/persona_translation_v4/merged/")
MAX_NEW_TOKENS = 64
SEED = 42
DTYPE = torch.float32
WORK = "/kaggle/working"

assert torch.cuda.device_count() >= 2, "this notebook needs two GPUs (Kaggle: GPU T4 x2)"
ENC_DEVICE, DEC_DEVICE = torch.device("cuda:0"), torch.device("cuda:1")
HF_TOKEN = HF_TOKEN.strip() or None
assert HF_TOKEN, "Gemma 3 is gated: paste a Hugging Face token in Cell 0"
login(HF_TOKEN)
api = HfApi()
REV = {r: api.model_info(r, token=HF_TOKEN).sha for r in (ENCODER, DECODER)}
REV.update({r: api.dataset_info(r).sha for r in (TOP_REPO, PHINC_REPO)})
print({"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
       "gpus": [torch.cuda.get_device_name(i) for i in range(2)], "revisions": REV})

## Cell 2 — Persona data (v4 merged)

Encoder input `"<persona name>: <English>"`; target the reference Hinglish line.

In [ ]:
# Cell 2 — v4 merged splits

def fetch_jsonl(name):
    return [json.loads(l) for l in urllib.request.urlopen(DATA + name).read().decode("utf-8").splitlines() if l]


MANIFEST = json.load(urllib.request.urlopen(DATA + "manifest.json"))
NAME = {p: v["name"] for p, v in MANIFEST["personas"].items()}
SPLITS = {s: [{**r, "persona": r["persona_id"], "enc_input": f"{NAME[r['persona_id']]}: {r['source']}"}
              for r in fetch_jsonl(f"{s}.jsonl")]
          for s in ("train", "validation", "test", "ood", "minimal_pairs")}
TRAIN, VAL = SPLITS["train"], SPLITS["validation"]
print({s: len(rs) for s, rs in SPLITS.items()})

## Cell 3 — Public corpora: download, clean, guard against leakage, sample

- **Hinglish-TOP:** training file; keep pairs with 3–30 words on both sides and no Devanagari.
- **PHINC:** remove @handles, links and #hashtags, then the same length and script rules, and an English/Hinglish length ratio between 0.5 and 2.
- Drop pairs whose English resembles any v4 evaluation sentence (word overlap ≥ 0.6); de-duplicate on the English.
- Hold out 200 pairs from each corpus; the training pool is Hinglish-TOP human pairs, cleaned PHINC and Hinglish-TOP synthetic pairs, shuffled and cut to `CORPUS_TOTAL` (default 12,000; 0 skips stage 1).

In [ ]:
# Cell 3 — Corpora

def download(repo, path, name):
    url = f"https://huggingface.co/datasets/{repo}/resolve/{REV[repo]}/{path}"
    local = f"/tmp/{name}"
    if not os.path.exists(local):
        urllib.request.urlretrieve(url, local)
    return local


top = pd.read_parquet(download(TOP_REPO, "data/train-00000-of-00001.parquet", "top_train.parquet"))
phinc = pd.read_csv(download(PHINC_REPO, "hinglish_english_parallel_corpus.csv", "phinc.csv"))
DEVA = re.compile(r"[ऀ-ॿ]")


def clean_tweet(t):
    t = re.sub(r"https?://\S+|www\.\S+", " ", str(t))
    t = re.sub(r"[@#]\w+", " ", t)
    return re.sub(r"\s+", " ", t).strip(" .…:-")


def usable(en, hi):
    ne, nh = len(en.split()), len(hi.split())
    return 3 <= ne <= 30 and 3 <= nh <= 30 and not DEVA.search(hi) and 0.5 <= nh / ne <= 2.0


pairs = {"top_human": [], "top_synthetic": [], "phinc": []}
for en, hi, gen in zip(top["en"], top["hi_en"], top["generated_by"]):
    en, hi = str(en).strip(), str(hi).strip()
    if usable(en, hi):
        pairs["top_human" if gen == "human" else "top_synthetic"].append((en, hi))
for hi, en in zip(phinc["Sentence"], phinc["English_Translation"]):
    en, hi = clean_tweet(en), clean_tweet(hi)
    if usable(en, hi):
        pairs["phinc"].append((en, hi))

words = lambda t: set(re.findall(r"[a-z0-9']+", t.lower()))
EVAL_WORDS = [words(r["source"]) for s in ("validation", "test", "ood", "minimal_pairs") for r in SPLITS[s]]


def leaks(en):
    w = words(en)
    return any(len(w & e) / len(w | e) >= 0.6 for e in EVAL_WORDS if w | e)


rng = random.Random(SEED)
for k in pairs:
    rng.shuffle(pairs[k])
seen, dropped = set(), Counter()


def take(items, n):
    """The first n usable pairs of a shuffled list: not seen before, not resembling a v4 evaluation sentence."""
    out = []
    for en, hi in items:
        if len(out) >= n:
            break
        key = en.lower()
        if key in seen:
            dropped["duplicate"] += 1
        elif leaks(en):
            dropped["resembles a v4 evaluation sentence"] += 1
        else:
            seen.add(key)
            out.append((en, hi))
    return out


CORPUS_TOTAL = 12000                       # stage-1 pairs; 0 skips stage 1
PHINC_MAX = CORPUS_TOTAL // 2              # balance: at most half from PHINC, the rest Hinglish-TOP
HELD = [("top", *p) for p in take(pairs["top_human"], 200)] + [("phinc", *p) for p in take(pairs["phinc"], 200)]
train_pairs = [("top", *p) for p in take(pairs["top_human"][200:], len(pairs["top_human"]))]
train_pairs += [("phinc", *p) for p in take(pairs["phinc"][200:], PHINC_MAX)]
train_pairs += [("top", *p) for p in take(pairs["top_synthetic"], max(0, CORPUS_TOTAL - len(train_pairs)))]
rng.shuffle(train_pairs)
train_pairs = train_pairs[:CORPUS_TOTAL]
print({k: len(v) for k, v in pairs.items()}, "usable before sampling | dropped while sampling", dict(dropped))


def corpus_record(src, en, hi):
    return {"corpus": src, "source": en, "target": hi, "enc_input": f"Plain: {en}", "references": [hi]}


CORPUS_TRAIN = [corpus_record(*p) for p in train_pairs]
CORPUS_HELD = [corpus_record(*p) for p in HELD]
print(f"stage 1 training pairs {len(CORPUS_TRAIN)} {dict(Counter(r['corpus'] for r in CORPUS_TRAIN))} | "
      f"held out {len(CORPUS_HELD)}")
for r in CORPUS_TRAIN[:4]:
    print(f"  {r['enc_input']}  ->  {r['target']}")

## Cell 4 — Models

In [ ]:
# Cell 4 — ModernBERT on GPU 0; Gemma 3 4B IT (4-bit) on GPU 1

from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig, Gemma3ForConditionalGeneration

used = {d: torch.cuda.memory_allocated(d) / 2**30 for d in (ENC_DEVICE, DEC_DEVICE)}
if max(used.values()) > 0.5:
    raise RuntimeError("GPU memory already in use: restart the kernel and run all cells once, from the top.")

enc_tok = AutoTokenizer.from_pretrained(ENCODER, revision=REV[ENCODER])
enc_cfg = AutoConfig.from_pretrained(ENCODER, revision=REV[ENCODER])
if hasattr(enc_cfg, "reference_compile"):
    enc_cfg.reference_compile = False
encoder = AutoModel.from_pretrained(ENCODER, revision=REV[ENCODER], config=enc_cfg, dtype=DTYPE,
                                    attn_implementation="sdpa").to(ENC_DEVICE)
dec_tok = AutoTokenizer.from_pretrained(DECODER, revision=REV[DECODER], token=HF_TOKEN)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
decoder = Gemma3ForConditionalGeneration.from_pretrained(DECODER, revision=REV[DECODER], dtype=DTYPE, token=HF_TOKEN,
                                                         quantization_config=quant, device_map={"": DEC_DEVICE.index},
                                                         attn_implementation="eager")
START = dec_tok.bos_token_id
EOS = dec_tok.convert_tokens_to_ids("<end_of_turn>")
eos = decoder.generation_config.eos_token_id
STOPS = sorted(set((eos if isinstance(eos, list) else [eos]) + [EOS]))
PAD = dec_tok.pad_token_id
D_ENC, D_DEC = encoder.config.hidden_size, decoder.config.text_config.hidden_size
LAYERS = next(m for n, m in decoder.named_modules() if n.endswith("language_model.layers"))
N_LAYERS = len(LAYERS)
print(f"encoder width {D_ENC} | decoder width {D_DEC}, {N_LAYERS} layers | start {START}, end {EOS}")

## Cell 5 — Bridge, LoRA, and the prefix forward pass

Each training row is `[<bos>] + prefix + target + <end_of_turn>` as input embeddings; the loss is on the target only. Translation passes `[<bos>] + prefix` to `generate`, which returns only the new tokens.

In [ ]:
# Cell 5 — Bridge (MLP into Gemma's embedding space), LoRA, the prefix forward pass, translation

import torch.nn.functional as F
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


class Bridge(nn.Module):
    """Encoder states -> vectors in Gemma's input-embedding space: a 2-layer MLP, then RMSNorm at the embeddings' scale."""

    def __init__(self, d_in, d_out):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(d_in, d_out), nn.GELU(), nn.Linear(d_out, d_out))
        self.norm = nn.RMSNorm(d_out)

    def forward(self, h):
        return self.norm(self.mlp(h))


torch.manual_seed(SEED)
bridge = Bridge(D_ENC, D_DEC).to(ENC_DEVICE)
decoder = prepare_model_for_kbit_training(decoder, use_gradient_checkpointing=True,
                                          gradient_checkpointing_kwargs={"use_reentrant": False})
PROJ = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
dec_model = get_peft_model(decoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                               target_modules=rf".*language_model.*\.({'|'.join(PROJ)})"))
encoder = get_peft_model(encoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["Wqkv", "Wo", "Wi"]))
EMBED = dec_model.get_input_embeddings()         # Gemma's own token embeddings (with Gemma's scaling)

with torch.no_grad():                             # start the prefix at the scale of real token embeddings
    sample = dec_tok(" ".join(r["target"] for r in TRAIN[:50]), add_special_tokens=False)["input_ids"]
    EMB_RMS = EMBED(torch.tensor(sample, device=DEC_DEVICE)).float().pow(2).mean(-1).sqrt().mean().item()
    bridge.norm.weight.fill_(EMB_RMS)

GROUPS = {"encoder LoRA": [p for n, p in encoder.named_parameters() if "lora_" in n],
          "bridge": list(bridge.parameters()),
          "decoder LoRA": [p for n, p in dec_model.named_parameters() if "lora_" in n]}
for params in GROUPS.values():
    for p in params:
        p.requires_grad_(True)
LRS = {"encoder LoRA": 1e-4, "bridge": 3e-4, "decoder LoRA": 5e-5}


def prefixes(texts):
    """Encoder input texts -> one prefix per text: bridge vectors for its real (unpadded) tokens, on GPU 1."""
    t = enc_tok(texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(ENC_DEVICE)
    h = encoder(input_ids=t["input_ids"], attention_mask=t["attention_mask"]).last_hidden_state
    p = bridge(h).to(DEC_DEVICE)
    return [p[i, :int(n)] for i, n in enumerate(t["attention_mask"].sum(1))]


def embed_ids(ids):
    return EMBED(torch.tensor(ids, device=DEC_DEVICE))


def batch_embeds(records, with_target=True):
    """Each row: [<bos>] + prefix + (target + <end_of_turn>), right-padded. Labels on the target only."""
    rows, labels = [], []
    for r, p in zip(records, prefixes([r["enc_input"] for r in records])):
        head = torch.cat([embed_ids([START]), p])
        if with_target:
            tgt = dec_tok(r["target"], add_special_tokens=False)["input_ids"][:96] + [EOS]
            rows.append(torch.cat([head, embed_ids(tgt)]))
            labels.append([-100] * len(head) + tgt)
        else:
            rows.append(head)
    T = max(len(x) for x in rows)
    emb = torch.stack([F.pad(x, (0, 0, 0, T - len(x))) for x in rows])
    mask = torch.tensor([[1] * len(x) + [0] * (T - len(x)) for x in rows], device=DEC_DEVICE)
    lab = torch.tensor([y + [-100] * (T - len(y)) for y in labels], device=DEC_DEVICE) if with_target else None
    return emb, mask, lab


def loss_on(records):
    emb, mask, lab = batch_embeds(records)
    return dec_model(inputs_embeds=emb, attention_mask=mask, labels=lab, token_type_ids=torch.zeros_like(mask)).loss


@torch.no_grad()
def translate(r):
    dec_model.eval()
    encoder.eval()
    emb, mask, _ = batch_embeds([r], with_target=False)
    out = dec_model.generate(inputs_embeds=emb, attention_mask=mask, do_sample=False, num_beams=1,
                             max_new_tokens=MAX_NEW_TOKENS, eos_token_id=STOPS, pad_token_id=PAD,
                             temperature=None, top_p=None, top_k=None)
    new = out[0].tolist()                         # with only inputs_embeds given, generate returns the new tokens
    return {"output": dec_tok.decode(new, skip_special_tokens=True).strip(), "stopped": bool(new) and new[-1] in STOPS}


@torch.no_grad()
def mean_loss(records, batch=8):
    dec_model.eval()
    encoder.eval()
    total = 0.0
    for i in range(0, len(records), batch):
        chunk = records[i:i + batch]
        total += loss_on(chunk).item() * len(chunk)
    return round(total / len(records), 4)


def make_optimizer(steps):
    opt = torch.optim.AdamW([{"params": GROUPS[g], "lr": lr} for g, lr in LRS.items()], weight_decay=0.0)
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / max(1, steps)))


def snapshot():
    return [p.detach().to("cpu", copy=True) for params in GROUPS.values() for p in params]


def restore(state):
    for p, s in zip([p for params in GROUPS.values() for p in params], state):
        p.data.copy_(s.to(p.device))


def chrf(hyp, ref, n_max=6, beta=2.0):
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    ps, rs = [], []
    for n in range(1, n_max + 1):
        hc = Counter(h[i:i + n] for i in range(len(h) - n + 1))
        rc = Counter(r[i:i + n] for i in range(len(r) - n + 1))
        if hc and rc:
            m = sum((hc & rc).values())
            ps.append(m / sum(hc.values()))
            rs.append(m / sum(rc.values()))
    if not ps:
        return 0.0
    p, q = statistics.mean(ps), statistics.mean(rs)
    return 100 * (1 + beta ** 2) * p * q / (beta ** 2 * p + q) if p + q else 0.0


def evaluate(records):
    rows = []
    for r in records:
        out = translate(r)
        rows.append({**{k: r.get(k) for k in ("id", "persona", "corpus", "source", "enc_input", "references")}, **out,
                     "chrf": max(chrf(out["output"], ref) for ref in r["references"])})
    score = {"n": len(rows), "chrf": round(statistics.mean(x["chrf"] for x in rows), 1),
             "stopped": round(sum(x["stopped"] for x in rows) / len(rows), 3),
             "distinct_outputs": len({x["output"] for x in rows})}
    return rows, score


def show(rows, n, seed=7):
    for x in random.Random(seed).sample(rows, min(n, len(rows))):
        print(f"  in : {x['enc_input']}\n  out: {x['output']}" + ("" if x["stopped"] else "  [did not stop]") +
              f"\n  ref: {x['references'][0]}\n")


print({k: f"{sum(p.numel() for p in v) / 1e6:.2f} M" for k, v in GROUPS.items()}, f"| prefix scale {EMB_RMS:.3f}")
print("before training:", translate(VAL[0])["output"][:120])

## Cell 6 — Stage 1: translation pre-training on the public corpora

One pass over `CORPUS_TOTAL` pairs (default 12,000), batches of 8, linear decay. Every 500 steps: held-out loss (400 pairs). Skipped when `CORPUS_TOTAL` is 0. After the pass: chrF, stopping and distinct outputs on 120 held-out pairs (60 from each corpus), with examples. The stage-1 weights are saved to `/kaggle/working/stage1.pt`.

In [ ]:
# Cell 6 — Stage 1

S1_BATCH, S1_EVAL_EVERY = 8, 500
s1_steps = -(-len(CORPUS_TRAIN) // S1_BATCH)
opt, sched = make_optimizer(s1_steps)
S1_LOG = [{"step": 0, "held_out_loss": mean_loss(CORPUS_HELD)}]
print(S1_LOG[-1])
torch.cuda.reset_peak_memory_stats(DEC_DEVICE)
t0, run = time.perf_counter(), []
for step in range(1, s1_steps + 1):
    dec_model.train()
    encoder.train()
    loss = loss_on(CORPUS_TRAIN[(step - 1) * S1_BATCH: step * S1_BATCH])
    loss.backward()
    opt.step()
    sched.step()
    opt.zero_grad()
    run.append(loss.item())
    if step % S1_EVAL_EVERY == 0 or step == s1_steps:
        S1_LOG.append({"step": step, "train_loss": round(sum(run) / len(run), 4),
                       "held_out_loss": mean_loss(CORPUS_HELD)})
        run = []
        print(S1_LOG[-1], f"| {time.perf_counter() - t0:.0f}s | peak GPU 1 "
              f"{torch.cuda.max_memory_allocated(DEC_DEVICE) / 2**30:.1f} GB")

torch.save(snapshot(), f"{WORK}/stage1.pt")
S1_PROBE = [r for r in CORPUS_HELD if r["corpus"] == "top"][:60] + [r for r in CORPUS_HELD if r["corpus"] == "phinc"][:60]
S1_ROWS, S1_SCORE = evaluate(S1_PROBE) if CORPUS_TRAIN else ([], None)
print("\nstage 1, held-out corpus pairs:", S1_SCORE)
if S1_ROWS:
    show(S1_ROWS, 8)

## Cell 7 — Stage 2: persona fine-tuning on v4 (continuing from stage 1)

Same as `7-simple-bridge/`: micro-batches of 4 with 2-step accumulation, up to 6 epochs, early stopping after 2 epochs without a lower v4 validation loss, best epoch kept. Each epoch prints three sample translations.

In [ ]:
# Cell 7 — Stage 2

MAX_EPOCHS, PATIENCE, MICRO, ACCUM = 6, 2, 4, 2
opt, sched = make_optimizer(MAX_EPOCHS * -(-len(TRAIN) // (MICRO * ACCUM)))
SAMPLES = [VAL[0], VAL[4], VAL[8]]
S2_LOG = [{"epoch": 0, "validation_loss": mean_loss(VAL)}]
print(S2_LOG[-1])
for r in SAMPLES:
    print(f"    {r['enc_input'][:70]:70s} -> {translate(r)['output']}")
best = (S2_LOG[0]["validation_loss"], 0, snapshot())
since_best, rng, t0 = 0, random.Random(SEED), time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    dec_model.train()
    encoder.train()
    order = rng.sample(TRAIN, len(TRAIN))
    batches = [order[i:i + MICRO] for i in range(0, len(order), MICRO)]
    run = []
    for n, b in enumerate(batches, 1):
        loss = loss_on(b)
        (loss / ACCUM).backward()
        run.append(loss.item())
        if n % ACCUM == 0 or n == len(batches):
            opt.step()
            sched.step()
            opt.zero_grad()
    S2_LOG.append({"epoch": epoch, "train_loss": round(sum(run) / len(run), 4), "validation_loss": mean_loss(VAL)})
    print(S2_LOG[-1], f"| {time.perf_counter() - t0:.0f}s")
    for r in SAMPLES:
        print(f"    {r['enc_input'][:70]:70s} -> {translate(r)['output']}")
    if S2_LOG[-1]["validation_loss"] < best[0]:
        best, since_best = (S2_LOG[-1]["validation_loss"], epoch, snapshot()), 0
    else:
        since_best += 1
    if since_best >= PATIENCE:
        print(f"early stop: no lower validation loss for {PATIENCE} epochs")
        break
restore(best[2])
BEST_EPOCH = best[1]
print(f"kept epoch {BEST_EPOCH} (validation loss {best[0]})")

## Cell 8 — Translate v4 validation, test and out-of-domain; scores and examples

chrF against the best reference, stopping, distinct outputs, and word-for-word copies of training targets. For comparison, `7-simple-bridge/` (cross-attention, no pre-training) reached validation chrF 12.0 with 5 distinct outputs of 75; `8-bridge-pretraining/` reports the cross-attention design with pre-training.

In [ ]:
# Cell 8 — v4 evaluation

train_targets = {r["target"].strip() for r in TRAIN}
RESULTS, SCORES = {}, {}
for split in ("validation", "test", "ood"):
    RESULTS[split], SCORES[split] = evaluate(SPLITS[split])
    SCORES[split]["copies_of_training_targets"] = sum(x["output"] in train_targets for x in RESULTS[split])
    print(split, SCORES[split])
print("\nexamples (validation, test, out-of-domain):")
show(RESULTS["validation"] + RESULTS["test"] + RESULTS["ood"], 12)

## Cell 9 — Save, settings, records

Both LoRAs and the bridge are saved under `/kaggle/working/` (download them if the result is worth keeping).

In [ ]:
# Cell 9 — Save the trained parts; settings and records

encoder.save_pretrained(f"{WORK}/encoder-lora")
dec_model.save_pretrained(f"{WORK}/decoder-lora")
torch.save({"bridge": bridge.state_dict(), "prefix_scale": EMB_RMS}, f"{WORK}/bridge.pt")
print(json.dumps({"revisions": REV, "corpus": {"train": dict(Counter(r["corpus"] for r in CORPUS_TRAIN)),
                                               "held_out": len(CORPUS_HELD), "dropped": dict(dropped)},
                  "lrs": LRS, "stage1": {"batch": S1_BATCH, "log": S1_LOG, "score": S1_SCORE},
                  "stage2": {"log": S2_LOG, "kept_epoch": BEST_EPOCH}, "scores": SCORES}, indent=1))
print("\n----- RECORDS (JSON lines) -----")
for x in S1_ROWS:
    print(json.dumps({"split": "corpus_held_out", **x}, ensure_ascii=False))
for split, rows in RESULTS.items():
    for x in rows:
        print(json.dumps({"split": split, **x}, ensure_ascii=False))
print("----- END RECORDS -----")